# Brain MRI Image Classification & Clinical Decision Making

This notebook reproduces and extends the Orange + Excel + LINGO workflow in Python:
1. **Stratified Dataset Split (75% Train / 25% Test)** on all 1,505 brain MRI images (`Normal`, `Glioma`, `Meningioma`, `Pituitary`) with a fixed reproducibility seed (`RANDOM_SEED = 42`).
2. **Pre-trained Inception v3 Image Embedding Extraction** (2,048-dimensional feature vectors, matching Orange's `Image Embedding` + `Select Columns` widgets).
3. **Three Tuned Classification Models**:
   - **Logistic Regression** (on Inception v3 embeddings, tuned via 5-fold Stratified CV)
   - **Support Vector Machine (SVM)** (on Inception v3 embeddings, tuned via 5-fold Stratified CV)
   - **Fine-Tuned Convolutional Neural Network (CNN)** operating directly on raw MRI images (replacing Orange's MLP on embeddings)
4. **Best Model Selection** based on **Multi-class Log-Loss** (strictly proper scoring rule for downstream Bayesian expected-cost decision making), reporting Accuracy, Macro-F1, Weighted-F1, ROC-AUC, Brier Score, and Glioma Recall.
5. **Unconstrained Decision Making (Deterministic Cost Matrix)** on all 377 test images (reproducing `aidm_exam_decision.xlsx`).
6. **Constrained Decision Making (Scaled MILP Optimization)** on all 377 test images (reproducing and scaling `Lingo_otimaldecision.lg4` with `scipy.optimize.milp` / HiGHS).
7. **Final Readable & Usable Results Tables (in English)** displayed inline and exported to `decision_making_results.xlsx` and `patient_level_decisions.csv`.

In [ ]:
import os
import random
import zipfile
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

# Reproducibility Seed
RANDOM_SEED = 42
os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)
random.seed(RANDOM_SEED)

import numpy as np
import pandas as pd
from scipy.optimize import milp, LinearConstraint, Bounds
from scipy.sparse import lil_matrix

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    log_loss,
    roc_auc_score,
    confusion_matrix,
    recall_score,
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.applications.inception_v3 import preprocess_input

np.random.seed(RANDOM_SEED)
tf.keras.utils.set_random_seed(RANDOM_SEED)

# Display settings for clean readable tables
pd.set_option("display.max_columns", 25)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

# Exact class order matching aidm_exam_decision.xlsx and Lingo_otimaldecision.lg4
CLASSES = ["Normal", "Glioma", "Meningioma", "Pituitary"]
FOLDER_TO_CLASS = {
    "512Normal": "Normal",
    "512Glioma": "Glioma",
    "512Meningioma": "Meningioma",
    "512Pituitary": "Pituitary",
}
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}

# Exact clinical actions, direct exam costs (€), and disutility Cost Matrix C(a, s)
ACTIONS = ["No further exam", "2nd-level exam", "Biopsy / Urgent care"]
EXAM_COST = np.array([0.0, 200.0, 1500.0], dtype=float)
COST_MATRIX = np.array([
    [0.0,     50000.0, 5000.0, 3000.0],  # A0: No further exam
    [200.0,   10000.0,  200.0,  200.0],  # A1: 2nd-level exam
    [1500.0,   1500.0, 1500.0, 1500.0],  # A2: Biopsy / Urgent care
], dtype=float)

# Image and training constants
IMG_SIZE = (299, 299)  # Native Inception v3 input resolution
BATCH_SIZE = 32

print(f"TensorFlow version : {tf.__version__}")
print(f"GPUs detected      : {len(tf.config.list_physical_devices('GPU'))}")
print(f"Random seed        : {RANDOM_SEED}")

## 1. Dataset Loading & Stratified 75% / 25% Split

The code automatically locates either an already-extracted `Raw Data/Raw` directory or `Raw Data.zip` in the working directory (skipping macOS `__MACOSX` metadata files).  
We perform an OS-independent, reproducible **stratified 75% Train / 25% Test split** (`random_state=42`):
- **Train set (75%)**: 1,128 images used exclusively for model training, cross-validation, and internal CNN validation.
- **Test set (25%)**: 377 images held out for unbiased model evaluation and downstream **Decision Making** (`PZ_001` to `PZ_377`).

In [ ]:
# Path configuration: supports either 'Raw Data/Raw' folder or 'Raw Data.zip'
RAW_DIR = Path("Raw Data/Raw")
ZIP_PATH = Path("Raw Data.zip")

if not RAW_DIR.exists():
    if ZIP_PATH.exists():
        print(f"Extracting {ZIP_PATH} (skipping __MACOSX metadata)...")
        with zipfile.ZipFile(ZIP_PATH, "r") as zf:
            for member in zf.namelist():
                if member.startswith("__MACOSX") or "/._" in member or member.endswith("/"):
                    continue
                zf.extract(member, ".")
        print("Extraction complete.")
    else:
        raise FileNotFoundError(
            "Dataset not found. Place 'Raw Data.zip' or the extracted 'Raw Data/Raw/' "
            "folder in the notebook working directory."
        )

# Collect all valid JPEG images
records = []
for folder_name, class_name in FOLDER_TO_CLASS.items():
    folder_path = RAW_DIR / folder_name
    if not folder_path.exists():
        raise FileNotFoundError(f"Expected class folder not found: {folder_path}")
    for img_path in sorted(folder_path.glob("*.jpg")):
        if img_path.name.startswith("._"):
            continue
        records.append({
            "filepath": str(img_path),
            "filename": img_path.name,
            "class_name": class_name,
            "label": CLASS_TO_IDX[class_name],
        })

df_all = pd.DataFrame(records).sort_values(["label", "filename"]).reset_index(drop=True)

# Stratified 75% Train / 25% Test split (exact reproducibility seed)
train_df, test_df = train_test_split(
    df_all,
    test_size=0.25,
    stratify=df_all["label"],
    random_state=RANDOM_SEED,
)
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

# Assign sequential Patient IDs to the test split (PZ_001 .. PZ_377)
test_df.insert(0, "patient_id", [f"PZ_{i:03d}" for i in range(1, len(test_df) + 1)])

# Table 1: Dataset Split Summary
split_rows = []
for cls in CLASSES:
    n_tot = int((df_all["class_name"] == cls).sum())
    n_tr = int((train_df["class_name"] == cls).sum())
    n_te = int((test_df["class_name"] == cls).sum())
    split_rows.append({
        "Class": cls,
        "Total Images": n_tot,
        "Total (%)": round(100.0 * n_tot / len(df_all), 2),
        "Train Set (75%)": n_tr,
        "Test Set (25%)": n_te,
        "Test Share (%)": round(100.0 * n_te / len(test_df), 2),
    })
split_rows.append({
    "Class": "TOTAL",
    "Total Images": len(df_all),
    "Total (%)": 100.00,
    "Train Set (75%)": len(train_df),
    "Test Set (25%)": len(test_df),
    "Test Share (%)": 100.00,
})
table_1_split = pd.DataFrame(split_rows)
display(table_1_split)

## 2. Image Embedding Extraction (Pre-trained Inception v3)

To replicate Orange's **Image Embedding** (Inception v3) and **Select Columns** widgets:
- Every MRI image is decoded as RGB, resized to `299 × 299`, and normalized to `[-1, 1]` via `inception_v3.preprocess_input`.
- Features are extracted from the global average pooling layer (`include_top=False, pooling='avg'`) of **Inception v3 pre-trained on ImageNet**, producing **2,048 embedding features** per image.
- All 2,048 embedding columns are retained as features for **Logistic Regression** and **SVM**, with `class_name` (`Normal`, `Glioma`, `Meningioma`, `Pituitary`) as the target.
- Extracted embeddings are cached locally to `inception_v3_embeddings.npz` so subsequent runs do not repeat extraction.

In [ ]:
def make_image_dataset(filepaths, labels=None, batch_size=BATCH_SIZE, shuffle=False, seed=RANDOM_SEED):
    """Build a memory-efficient tf.data.Dataset returning float32 images in [0, 255]."""
    def _load_image(path):
        img_bytes = tf.io.read_file(path)
        img = tf.image.decode_jpeg(img_bytes, channels=3)
        img = tf.image.resize(img, IMG_SIZE, method="bilinear")
        return tf.cast(img, tf.float32)

    if labels is not None:
        ds = tf.data.Dataset.from_tensor_slices((filepaths, labels))
        if shuffle:
            ds = ds.shuffle(buffer_size=len(filepaths), seed=seed, reshuffle_each_iteration=True)
        ds = ds.map(lambda p, y: (_load_image(p), y), num_parallel_calls=tf.data.AUTOTUNE)
    else:
        ds = tf.data.Dataset.from_tensor_slices(filepaths)
        ds = ds.map(_load_image, num_parallel_calls=tf.data.AUTOTUNE)

    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)


EMBEDDING_CACHE = Path("inception_v3_embeddings.npz")

if EMBEDDING_CACHE.exists():
    print(f"Loading cached Inception v3 embeddings from {EMBEDDING_CACHE}...")
    cached = np.load(EMBEDDING_CACHE)
    X_train_emb = cached["X_train_emb"]
    X_test_emb = cached["X_test_emb"]
else:
    print("Loading pre-trained Inception v3 (ImageNet) embedder...")
    embedder = InceptionV3(weights="imagenet", include_top=False, pooling="avg", input_shape=(299, 299, 3))

    ds_train_emb = make_image_dataset(train_df["filepath"].values).map(
        lambda x: preprocess_input(x), num_parallel_calls=tf.data.AUTOTUNE
    )
    ds_test_emb = make_image_dataset(test_df["filepath"].values).map(
        lambda x: preprocess_input(x), num_parallel_calls=tf.data.AUTOTUNE
    )

    print("Extracting 2,048-dim embeddings for Train set (1,128 images)...")
    X_train_emb = embedder.predict(ds_train_emb, verbose=1)
    print("Extracting 2,048-dim embeddings for Test set (377 images)...")
    X_test_emb = embedder.predict(ds_test_emb, verbose=1)

    np.savez_compressed(EMBEDDING_CACHE, X_train_emb=X_train_emb, X_test_emb=X_test_emb)
    print(f"Saved embeddings to {EMBEDDING_CACHE}.")

y_train = train_df["label"].values
y_test = test_df["label"].values

assert not np.isnan(X_train_emb).any() and not np.isnan(X_test_emb).any(), "NaNs detected in embeddings!"
print(f"X_train_emb shape: {X_train_emb.shape} | X_test_emb shape: {X_test_emb.shape}")

## 3. Models & Hyperparameter Modifications (vs. Orange Defaults)

We train the same three model families used in the Orange workflow, applying targeted architectural and hyperparameter improvements designed for **brain MRI classification** and **calibrated probability estimation**:

### 1. Logistic Regression (on Inception v3 Embeddings)
- **Orange defaults**: L2 penalty, fixed `C = 1.0`, `max_iter = 100`, no explicit feature standardization pipeline, unweighted classes.
- **Modifications**:
  - Wrapped in a `Pipeline` with **`StandardScaler()`** fitted strictly inside each training fold (zero data leakage) so L2 regularization penalizes all 2,048 embedding dimensions uniformly.
  - Increased **`max_iter = 2000`** with solver **`lbfgs`** to guarantee full convergence on 2,048 features.
  - Set **`class_weight = 'balanced'`** to prevent mild majority-class bias (`Normal` has 396 images vs. 363 `Meningioma`).
  - Tuned regularization strength **`C ∈ {0.01, 0.1, 1.0, 10.0}`** via **5-fold Stratified Cross-Validation** optimizing **`neg_log_loss`** (directly improving probability calibration for the Cost Matrix).

### 2. Support Vector Machine — SVM (on Inception v3 Embeddings)
- **Orange defaults**: RBF kernel, fixed `C = 1.0`, `gamma = 'auto'`, `tol = 0.001`, iteration limit `100`.
- **Modifications**:
  - Wrapped in a `Pipeline` with **`StandardScaler()`** (critical for distance-based RBF kernels and margin stability).
  - Removed the artificial 100-iteration cap (`max_iter = -1`) so the dual QP solver converges to exact optimality.
  - Enabled **`probability = True`** (Platt scaling via internal 5-fold CV) and **`class_weight = 'balanced'`**.
  - Tuned **`kernel ∈ {'rbf', 'linear'}`** and **`C ∈ {0.1, 1.0, 10.0}`** (with `gamma = 'scale'` for RBF) via **5-fold Stratified CV** optimizing **`neg_log_loss`**.

### 3. Convolutional Neural Network — CNN (Replacing Orange's MLP on Embeddings)
- **Orange default**: A shallow Multi-Layer Perceptron (`100` hidden units, ReLU, Adam, `alpha=0.0001`, `max_iter=200`) trained on **frozen** ImageNet embeddings. Because ImageNet embeddings were learned on natural photographs rather than medical radiology, the convolutional filters cannot adapt to subtle MRI tumor boundaries, edema, or sellar/dural attachment cues.
- **Modifications (End-to-End Fine-Tuned CNN directly on MRI Images)**:
  - **Direct Image Input & MRI-Specific Data Augmentation**: Operates directly on raw `299 × 299 × 3` MRI images with anatomically plausible augmentations (`RandomFlip('horizontal')`, `RandomRotation(±14°)`, `RandomZoom(±10%)`, `RandomContrast(±10%)`). Vertical flips are intentionally excluded because brain anatomy is not vertically symmetric.
  - **Convolutional Backbone**: Pre-trained **InceptionV3** (`include_top=False`) with exact built-in pixel rescaling (`Rescaling(1/127.5, offset=-1.0)`).
  - **Regularized Classification Head**: `GlobalAveragePooling2D → BatchNormalization → Dropout(0.4) → Dense(256, ReLU, L2=1e-4) → BatchNormalization → Dropout(0.3) → Dense(4, Softmax)` to control overfitting on ~1,000 medical images.
  - **Leak-Free Internal Validation Split**: The 75% Train set (`1,128` images) is split internally 85% / 15% stratified (`958` train / `170` validation) for `EarlyStopping` and `ReduceLROnPlateau`; the 25% Test set (`377` images) is never seen during training.
  - **Two-Phase Transfer Learning**:
    - *Phase 1 (Warm-Up)*: Convolutional backbone frozen; train only the new head with `Adam(lr=1e-3)` up to 15 epochs so random head weights do not corrupt pre-trained filters.
    - *Phase 2 (Domain Fine-Tuning)*: Unfreeze the top Inception blocks (`mixed8` through `mixed10`, keeping all `BatchNormalization` layers frozen for stability with small mini-batches) and fine-tune end-to-end with a low learning rate `Adam(lr=1e-5)` up to 20 epochs.
  - **Probability-Preserving Loss**: Uses standard **`sparse_categorical_crossentropy`** (without label smoothing) together with balanced class weights so the softmax outputs remain calibrated posterior probabilities for the asymmetric Cost Matrix.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

lr_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        solver="lbfgs",
        max_iter=2000,
        class_weight="balanced",
        random_state=RANDOM_SEED,
    )),
])

lr_grid = GridSearchCV(
    estimator=lr_pipe,
    param_grid={"clf__C": [0.01, 0.1, 1.0, 10.0]},
    cv=cv,
    scoring="neg_log_loss",
    n_jobs=-1,
    refit=True,
)
lr_grid.fit(X_train_emb, y_train)
best_lr = lr_grid.best_estimator_
probs_lr = best_lr.predict_proba(X_test_emb)

print("Logistic Regression — Best params:", lr_grid.best_params_)
print(f"Logistic Regression — Best 5-fold CV Log-Loss: {-lr_grid.best_score_:.4f}")

In [ ]:
svm_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", SVC(
        probability=True,
        class_weight="balanced",
        random_state=RANDOM_SEED,
    )),
])

svm_param_grid = [
    {"clf__kernel": ["rbf"], "clf__C": [0.1, 1.0, 10.0], "clf__gamma": ["scale"]},
    {"clf__kernel": ["linear"], "clf__C": [0.1, 1.0, 10.0]},
]

svm_grid = GridSearchCV(
    estimator=svm_pipe,
    param_grid=svm_param_grid,
    cv=cv,
    scoring="neg_log_loss",
    n_jobs=-1,
    refit=True,
)
svm_grid.fit(X_train_emb, y_train)
best_svm = svm_grid.best_estimator_
probs_svm = best_svm.predict_proba(X_test_emb)

print("SVM — Best params:", svm_grid.best_params_)
print(f"SVM — Best 5-fold CV Log-Loss: {-svm_grid.best_score_:.4f}")

In [ ]:
# Internal 85/15 stratified split strictly inside the 75% Train set (Test set untouched)
cnn_train_df, cnn_val_df = train_test_split(
    train_df,
    test_size=0.15,
    stratify=train_df["label"],
    random_state=RANDOM_SEED,
)

ds_cnn_train = make_image_dataset(
    cnn_train_df["filepath"].values,
    cnn_train_df["label"].values,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=RANDOM_SEED,
)
ds_cnn_val = make_image_dataset(
    cnn_val_df["filepath"].values,
    cnn_val_df["label"].values,
    batch_size=BATCH_SIZE,
    shuffle=False,
)
ds_cnn_test = make_image_dataset(
    test_df["filepath"].values,
    labels=None,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

# Balanced class weights computed strictly on cnn_train_df
cw_vals = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(len(CLASSES)),
    y=cnn_train_df["label"].values,
)
class_weights_dict = {i: float(w) for i, w in enumerate(cw_vals)}

# MRI-appropriate data augmentation (no vertical flip)
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal", seed=RANDOM_SEED),
    layers.RandomRotation(0.04, seed=RANDOM_SEED),
    layers.RandomZoom(0.10, seed=RANDOM_SEED),
    layers.RandomContrast(0.10, seed=RANDOM_SEED),
], name="mri_data_augmentation")

# Build CNN architecture
tf.keras.utils.set_random_seed(RANDOM_SEED)
base_cnn = InceptionV3(weights="imagenet", include_top=False, input_shape=(299, 299, 3))
base_cnn.trainable = False

inputs = keras.Input(shape=(299, 299, 3), name="mri_input")
x = data_augmentation(inputs)
x = layers.Rescaling(scale=1.0 / 127.5, offset=-1.0, name="inception_rescaling")(x)
x = base_cnn(x, training=False)
x = layers.GlobalAveragePooling2D(name="global_avg_pool")(x)
x = layers.BatchNormalization(name="head_bn_1")(x)
x = layers.Dropout(0.40, name="head_dropout_1")(x)
x = layers.Dense(256, activation="relu", kernel_regularizer=regularizers.l2(1e-4), name="head_dense_256")(x)
x = layers.BatchNormalization(name="head_bn_2")(x)
x = layers.Dropout(0.30, name="head_dropout_2")(x)
outputs = layers.Dense(len(CLASSES), activation="softmax", name="class_probabilities")(x)

cnn_model = keras.Model(inputs, outputs, name="InceptionV3_MRI_CNN")

# --- Phase 1: Classification Head Warm-Up (Backbone Frozen) ---
cnn_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
callbacks_p1 = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2, min_lr=1e-6),
]

print("CNN Phase 1: Training custom head (frozen convolutional backbone)...")
history_p1 = cnn_model.fit(
    ds_cnn_train,
    validation_data=ds_cnn_val,
    epochs=15,
    class_weight=class_weights_dict,
    callbacks=callbacks_p1,
    verbose=1,
)

# --- Phase 2: End-to-End Fine-Tuning (Unfreeze mixed8 .. mixed10, keep BN frozen) ---
base_cnn.trainable = True
fine_tune_start_idx = [l.name for l in base_cnn.layers].index("mixed8")
for idx, layer in enumerate(base_cnn.layers):
    if idx < fine_tune_start_idx or isinstance(layer, layers.BatchNormalization):
        layer.trainable = False
    else:
        layer.trainable = True

cnn_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
callbacks_p2 = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2, min_lr=1e-7),
]

print("CNN Phase 2: Fine-tuning top Inception blocks (mixed8-mixed10)...")
history_p2 = cnn_model.fit(
    ds_cnn_train,
    validation_data=ds_cnn_val,
    epochs=20,
    class_weight=class_weights_dict,
    callbacks=callbacks_p2,
    verbose=1,
)

# Predict calibrated class probabilities on the 377 Test images
probs_cnn = cnn_model.predict(ds_cnn_test, verbose=1)
probs_cnn = probs_cnn / probs_cnn.sum(axis=1, keepdims=True)  # ensure exact row sum = 1.0
print("CNN test probability matrix shape:", probs_cnn.shape)

## 4. Model Evaluation & Best Model Selection

### Why Multi-class Log-Loss is the Primary Selection Criterion
In standard classification, **Accuracy** evaluates only the argmax class label and treats all errors identically. However, in this clinical decision-making pipeline, the **full probability vector** $P(i, s)$ enters the asymmetric expected cost calculation:

$$\text{EC}(i, a) = \sum_{s=0}^{3} P(i, s) \cdot C(a, s)$$

where missing a **Glioma** (`No further exam`) carries a **€50,000** penalty versus **€200** for a 2nd-level MRI and **€1,500** for a Biopsy. Even a small probability calibration error (e.g., predicting $P(\text{Glioma}) = 0.01$ vs. $0.05$) shifts the optimal clinical action.

- **Multi-class Log-Loss (Cross-Entropy)** is a **strictly proper scoring rule**: it is uniquely minimized when predicted probabilities match true posterior probabilities and heavily penalizes overconfident misclassifications.
- Therefore, we select the model with the **lowest Test Multi-class Log-Loss** to feed into the Cost Matrix and MILP optimizer, while also reporting **Accuracy**, **Macro-F1**, **Weighted-F1**, **ROC-AUC (OvR Macro)**, **Multi-class Brier Score**, and **Glioma Recall** (critical clinical safety metric).

In [ ]:
def multiclass_brier_score(y_true, y_prob, n_classes=4):
    y_onehot = np.eye(n_classes)[y_true]
    return float(np.mean(np.sum((y_prob - y_onehot) ** 2, axis=1)))


model_probs = {
    "Logistic Regression (Tuned, InceptionV3 Emb.)": probs_lr,
    "SVM (Tuned, InceptionV3 Emb.)": probs_svm,
    "CNN (Fine-Tuned InceptionV3 on Raw MRI)": probs_cnn,
}

eval_rows = []
for name, probs in model_probs.items():
    preds = np.argmax(probs, axis=1)
    glioma_idx = CLASS_TO_IDX["Glioma"]
    eval_rows.append({
        "Model": name,
        "Log-Loss (Primary, Lower=Better)": log_loss(y_test, probs, labels=list(range(len(CLASSES)))),
        "Accuracy": accuracy_score(y_test, preds),
        "Macro-F1": f1_score(y_test, preds, average="macro"),
        "Weighted-F1": f1_score(y_test, preds, average="weighted"),
        "ROC-AUC (OvR Macro)": roc_auc_score(y_test, probs, multi_class="ovr", average="macro"),
        "Brier Score (Lower=Better)": multiclass_brier_score(y_test, probs, n_classes=len(CLASSES)),
        "Glioma Recall": recall_score(y_test == glioma_idx, preds == glioma_idx),
    })

table_2_models = pd.DataFrame(eval_rows).sort_values("Log-Loss (Primary, Lower=Better)").reset_index(drop=True)
best_model_name = table_2_models.loc[0, "Model"]
table_2_models["Selected"] = np.where(table_2_models["Model"] == best_model_name, "YES (Best)", "No")

best_probs = model_probs[best_model_name]
best_preds = np.argmax(best_probs, axis=1)

# Table 3: Best Model Confusion Matrix (Test Set, N = 377)
cm = confusion_matrix(y_test, best_preds, labels=list(range(len(CLASSES))))
table_3_cm = pd.DataFrame(
    cm,
    index=[f"True: {c}" for c in CLASSES],
    columns=[f"Pred: {c}" for c in CLASSES],
)
table_3_cm["Total"] = table_3_cm.sum(axis=1)
table_3_cm.loc["Total"] = table_3_cm.sum(axis=0)

print(f"Selected Best Model (Lowest Test Log-Loss): {best_model_name}\n")
display(table_2_models)
display(table_3_cm)

## 5. Unconstrained Decision Making (Deterministic Cost Matrix)

Using the exact Cost / Disutility Matrix $C(a, s)$ from `aidm_exam_decision.xlsx` and `Lingo_otimaldecision.lg4`, for each test patient $i \in \{1, \dots, 377\}$ and each clinical action $a \in \{0, 1, 2\}$ we compute:

$$\text{EC}(i, a) = \sum_{s=0}^{3} P(i, s) \cdot C(a, s)$$

and deterministically select the action minimizing expected clinical & economic disutility:

$$a^*_{\text{unc}}(i) = \arg\min_{a \in \{0, 1, 2\}} \text{EC}(i, a)$$

In [ ]:
# Table 4: Cost / Disutility Matrix & Direct Exam Costs (€)
table_4_cost_matrix = pd.DataFrame(
    COST_MATRIX,
    index=ACTIONS,
    columns=CLASSES,
)
table_4_cost_matrix.insert(0, "Direct Exam Cost (€)", EXAM_COST)
table_4_cost_matrix.index.name = "Clinical Action"

# Expected Cost matrix EC(i, a) for all 377 test patients
EC_matrix = best_probs @ COST_MATRIX.T  # shape: (377, 3)

# Deterministic unconstrained optimal action
unc_action_idx = np.argmin(EC_matrix, axis=1)
unc_exp_cost = EC_matrix[np.arange(len(test_df)), unc_action_idx]
unc_direct_cost = EXAM_COST[unc_action_idx]

print(f"Unconstrained Total Expected Disutility : €{unc_exp_cost.sum():,.2f}")
print(f"Unconstrained Total Direct Exam Spend   : €{unc_direct_cost.sum():,.2f}")
display(table_4_cost_matrix)

## 6. Constrained Decision Making (Scaled MILP Optimization)

We formulate the exact Binary Integer Linear Program (MILP) from `Lingo_otimaldecision.lg4` over all $N = 377$ test patients ($377 \times 3 = 1,131$ binary decision variables $x_{i,a} \in \{0, 1\}$), solved to global optimality with `scipy.optimize.milp` (HiGHS solver):

$$\min \sum_{i=1}^{377} \sum_{a=0}^{2} \text{EC}(i, a) \cdot x_{i, a}$$

subject to:
1. **Single Assignment**: $\sum_{a=0}^{2} x_{i, a} = 1 \quad \forall i = 1, \dots, 377$
2. **Hospital Budget**: $\sum_{i=1}^{377} \sum_{a=0}^{2} \text{EXAM\_COST}(a) \cdot x_{i, a} \le \text{MAX\_BUDGET}$
3. **2nd-Level MRI Capacity**: $\sum_{i=1}^{377} x_{i, 1} \le \text{MAX\_MRI\_SLOTS}$
4. **Biopsy / Urgent Care Capacity**: $\sum_{i=1}^{377} x_{i, 2} \le \text{MAX\_BIOPSY\_SLOTS}$

### Constraint Scaling Rationale ($N = 10 \rightarrow N = 377$, scale factor $= 37.7\times$)
- **Slot Capacities (Exact Proportional Scaling of LINGO Ratios)**:
  - In LINGO ($N = 10$), `MAX_MRI_SLOTS = 3` (30% of patients) and `MAX_BIOPSY_SLOTS = 2` (20% of patients).
  - Scaling proportionally to $N = 377$ gives **`MAX_MRI_SLOTS = 113`** ($0.30 \times 377 = 113.1$) and **`MAX_BIOPSY_SLOTS = 75`** ($0.20 \times 377 = 75.4$).
  - Relative to the test-set clinical pathology mix (184 Meningioma + Pituitary cases and 94 Glioma cases), these slot caps cover ~61% of MRI demand and ~80% of Biopsy demand, creating realistic hospital capacity triage.
- **Budget Constraint (`MAX_BUDGET = €120,000`)**:
  - In the original LINGO file, full slot utilization spent only $3 \times \text{€}200 + 2 \times \text{€}1{,}500 = \text{€}3{,}600 < \text{€}5{,}000$, making the budget constraint mathematically non-binding (slack).
  - Naive linear scaling ($\text{€}5{,}000 \times 37.7 = \text{€}188{,}500$) would repeat that redundancy because full slot utilization on 377 patients costs at most $113 \times \text{€}200 + 75 \times \text{€}1{,}500 = \text{€}135{,}100$.
  - Setting **`MAX_BUDGET = €120,000`** (~88.8% of the €135,100 slot-saturated ceiling, ~€318/patient) ensures that **budget and slot constraints interact actively**, forcing true clinical cost-effectiveness trade-offs between expensive Biopsies (€1,500) and high-leverage 2nd-level MRIs (€200). Feasibility is always guaranteed because `No further exam` has zero direct cost and no slot cap.

In [ ]:
N_TEST = len(test_df)
N_ACTIONS = len(ACTIONS)

# Scaled constraints for N = 377 test patients
MAX_MRI_SLOTS = round(0.30 * N_TEST)     # 113 slots (30% of test batch, exact LINGO ratio)
MAX_BIOPSY_SLOTS = round(0.20 * N_TEST)  # 75 slots  (20% of test batch, exact LINGO ratio)
MAX_BUDGET = 120000.0                    # €120,000  (active budget below €135,100 slot-saturated ceiling)

# Build MILP matrices
n_vars = N_TEST * N_ACTIONS
c_obj = EC_matrix.flatten()

A = lil_matrix((N_TEST + 3, n_vars), dtype=float)
lb = np.full(N_TEST + 3, -np.inf, dtype=float)
ub = np.full(N_TEST + 3, np.inf, dtype=float)

# Constraint 1: Exactly one action per patient
for i in range(N_TEST):
    A[i, i * N_ACTIONS : (i + 1) * N_ACTIONS] = 1.0
    lb[i] = 1.0
    ub[i] = 1.0

# Constraint 2: Maximum total monetary budget (€)
A[N_TEST, :] = np.tile(EXAM_COST, N_TEST)
ub[N_TEST] = MAX_BUDGET

# Constraint 3: Maximum 2nd-level MRI slots (action index 1)
A[N_TEST + 1, 1::N_ACTIONS] = 1.0
ub[N_TEST + 1] = MAX_MRI_SLOTS

# Constraint 4: Maximum Biopsy / Urgent care slots (action index 2)
A[N_TEST + 2, 2::N_ACTIONS] = 1.0
ub[N_TEST + 2] = MAX_BIOPSY_SLOTS

# Solve Binary ILP to global optimality (HiGHS)
milp_res = milp(
    c=c_obj,
    integrality=np.ones(n_vars, dtype=int),
    bounds=Bounds(0.0, 1.0),
    constraints=LinearConstraint(A.tocsc(), lb, ub),
)
if not milp_res.success:
    raise RuntimeError(f"MILP optimization failed: {milp_res.message}")

X_opt = np.round(milp_res.x).astype(int).reshape(N_TEST, N_ACTIONS)
con_action_idx = np.argmax(X_opt, axis=1)
con_exp_cost = EC_matrix[np.arange(N_TEST), con_action_idx]
con_direct_cost = EXAM_COST[con_action_idx]

# Programmatic verification of all MILP constraints
assert np.all(X_opt.sum(axis=1) == 1), "Assignment constraint violated!"
assert con_direct_cost.sum() <= MAX_BUDGET + 1e-6, "Budget constraint violated!"
assert int((con_action_idx == 1).sum()) <= MAX_MRI_SLOTS, "MRI slot constraint violated!"
assert int((con_action_idx == 2).sum()) <= MAX_BIOPSY_SLOTS, "Biopsy slot constraint violated!"

print(f"MILP Solver Status                    : {milp_res.message}")
print(f"Constrained Total Expected Disutility : €{con_exp_cost.sum():,.2f}")
print(f"Constrained Total Direct Exam Spend   : €{con_direct_cost.sum():,.2f} / €{MAX_BUDGET:,.2f}")
print(f"2nd-level MRI Slots Used              : {(con_action_idx == 1).sum()} / {MAX_MRI_SLOTS}")
print(f"Biopsy / Urgent Care Slots Used       : {(con_action_idx == 2).sum()} / {MAX_BIOPSY_SLOTS}")

## 7. Final Readable & Usable Results Tables (English)

Below are all final summary and patient-level tables in English. All tables are also exported automatically to:
- **`decision_making_results.xlsx`** (multi-sheet Excel workbook with auto-sized columns containing every table)
- **`patient_level_decisions.csv`** (complete 377-row patient-level table)

In [ ]:
# Table 5: Unconstrained vs. Constrained Decision-Making Summary (N = 377)
summary_rows = []
for a_idx, a_name in enumerate(ACTIONS):
    u_mask = unc_action_idx == a_idx
    c_mask = con_action_idx == a_idx
    summary_rows.append({
        "Prescribed Action": a_name,
        "Unconstrained Patients": int(u_mask.sum()),
        "Unconstrained (%)": round(100.0 * u_mask.mean(), 2),
        "Unconstrained Direct Cost (€)": float(unc_direct_cost[u_mask].sum()),
        "Unconstrained Expected Disutility (€)": float(unc_exp_cost[u_mask].sum()),
        "Constrained Patients": int(c_mask.sum()),
        "Constrained (%)": round(100.0 * c_mask.mean(), 2),
        "Constrained Direct Cost (€)": float(con_direct_cost[c_mask].sum()),
        "Constrained Expected Disutility (€)": float(con_exp_cost[c_mask].sum()),
    })

summary_rows.append({
    "Prescribed Action": "TOTAL",
    "Unconstrained Patients": N_TEST,
    "Unconstrained (%)": 100.00,
    "Unconstrained Direct Cost (€)": float(unc_direct_cost.sum()),
    "Unconstrained Expected Disutility (€)": float(unc_exp_cost.sum()),
    "Constrained Patients": N_TEST,
    "Constrained (%)": 100.00,
    "Constrained Direct Cost (€)": float(con_direct_cost.sum()),
    "Constrained Expected Disutility (€)": float(con_exp_cost.sum()),
})
table_5_dm_summary = pd.DataFrame(summary_rows)

# Table 6: Resource Constraints & Utilization Summary
mri_used = int((con_action_idx == 1).sum())
bio_used = int((con_action_idx == 2).sum())
bud_used = float(con_direct_cost.sum())

table_6_constraints = pd.DataFrame([
    {
        "Resource Constraint": "Hospital Budget (€)",
        "LINGO Original (N=10)": "€5,000",
        "Scaled Limit (N=377)": f"€{MAX_BUDGET:,.0f}",
        "Unconstrained Demand": f"€{unc_direct_cost.sum():,.0f}",
        "Constrained Usage": f"€{bud_used:,.0f}",
        "Utilization (%)": round(100.0 * bud_used / MAX_BUDGET, 2),
        "Slack Remaining": f"€{MAX_BUDGET - bud_used:,.0f}",
    },
    {
        "Resource Constraint": "2nd-Level MRI Slots",
        "LINGO Original (N=10)": "3 (30%)",
        "Scaled Limit (N=377)": f"{MAX_MRI_SLOTS} (30%)",
        "Unconstrained Demand": f"{int((unc_action_idx == 1).sum())}",
        "Constrained Usage": f"{mri_used}",
        "Utilization (%)": round(100.0 * mri_used / MAX_MRI_SLOTS, 2),
        "Slack Remaining": f"{MAX_MRI_SLOTS - mri_used}",
    },
    {
        "Resource Constraint": "Biopsy / Urgent Care Slots",
        "LINGO Original (N=10)": "2 (20%)",
        "Scaled Limit (N=377)": f"{MAX_BIOPSY_SLOTS} (20%)",
        "Unconstrained Demand": f"{int((unc_action_idx == 2).sum())}",
        "Constrained Usage": f"{bio_used}",
        "Utilization (%)": round(100.0 * bio_used / MAX_BIOPSY_SLOTS, 2),
        "Slack Remaining": f"{MAX_BIOPSY_SLOTS - bio_used}",
    },
])

# Table 7: Decision Reallocation Matrix (Unconstrained -> Constrained)
realloc_cm = confusion_matrix(unc_action_idx, con_action_idx, labels=[0, 1, 2])
table_7_reallocation = pd.DataFrame(
    realloc_cm,
    index=[f"Unconstrained: {a}" for a in ACTIONS],
    columns=[f"Constrained: {a}" for a in ACTIONS],
)
table_7_reallocation["Total"] = table_7_reallocation.sum(axis=1)
table_7_reallocation.loc["Total"] = table_7_reallocation.sum(axis=0)

# Table 8: Complete Patient-Level Results (All 377 Test Patients)
table_8_patients = pd.DataFrame({
    "Patient ID": test_df["patient_id"].values,
    "Image Filename": test_df["filename"].values,
    "True Class": test_df["class_name"].values,
    "Predicted Class": [CLASSES[i] for i in best_preds],
    "P(Normal)": best_probs[:, 0],
    "P(Glioma)": best_probs[:, 1],
    "P(Meningioma)": best_probs[:, 2],
    "P(Pituitary)": best_probs[:, 3],
    "Prob. Sum": best_probs.sum(axis=1),
    "EC(No Exam) (€)": EC_matrix[:, 0],
    "EC(2nd-Level MRI) (€)": EC_matrix[:, 1],
    "EC(Biopsy) (€)": EC_matrix[:, 2],
    "Unconstrained Decision": [ACTIONS[i] for i in unc_action_idx],
    "Constrained Decision": [ACTIONS[i] for i in con_action_idx],
    "Decision Changed": np.where(unc_action_idx != con_action_idx, "Yes", "No"),
    "Unconstrained Expected Cost (€)": unc_exp_cost,
    "Constrained Expected Cost (€)": con_exp_cost,
    "Additional Expected Disutility (€)": con_exp_cost - unc_exp_cost,
})

print("=" * 90)
print("TABLE 1: DATASET SPLIT SUMMARY")
print("=" * 90)
display(table_1_split)

print("\n" + "=" * 90)
print("TABLE 2: MODEL PERFORMANCE COMPARISON (TEST SET, N = 377)")
print("=" * 90)
display(table_2_models)

print("\n" + "=" * 90)
print(f"TABLE 3: CONFUSION MATRIX — {best_model_name.upper()} (TEST SET, N = 377)")
print("=" * 90)
display(table_3_cm)

print("\n" + "=" * 90)
print("TABLE 4: COST / DISUTILITY MATRIX & DIRECT EXAM COSTS (€)")
print("=" * 90)
display(table_4_cost_matrix)

print("\n" + "=" * 90)
print("TABLE 5: UNCONSTRAINED VS. CONSTRAINED DECISION-MAKING SUMMARY (N = 377)")
print("=" * 90)
display(table_5_dm_summary)

print("\n" + "=" * 90)
print("TABLE 6: RESOURCE CONSTRAINTS & UTILIZATION SUMMARY")
print("=" * 90)
display(table_6_constraints)

print("\n" + "=" * 90)
print("TABLE 7: DECISION REALLOCATION MATRIX (UNCONSTRAINED -> CONSTRAINED)")
print("=" * 90)
display(table_7_reallocation)

print("\n" + "=" * 90)
print("TABLE 8: PATIENT-LEVEL PREDICTIONS & OPTIMAL DECISIONS (FIRST 20 OF 377 PATIENTS)")
print("=" * 90)
display(table_8_patients.head(20))
print(f"Total patients with changed decision due to constraints: "
      f"{int((unc_action_idx != con_action_idx).sum())} / {N_TEST} "
      f"({100.0 * (unc_action_idx != con_action_idx).mean():.2f}%)")
print(f"Total expected disutility increase due to constraints : "
      f"€{float((con_exp_cost - unc_exp_cost).sum()):,.2f}")

# Export all tables to Excel (with auto-fitted column widths) and CSV
EXCEL_OUT = "decision_making_results.xlsx"
CSV_OUT = "patient_level_decisions.csv"

with pd.ExcelWriter(EXCEL_OUT, engine="openpyxl") as writer:
    sheets = [
        ("1_Dataset_Split", table_1_split, False),
        ("2_Model_Comparison", table_2_models, False),
        ("3_Confusion_Matrix", table_3_cm, True),
        ("4_Cost_Matrix", table_4_cost_matrix, True),
        ("5_Decision_Summary", table_5_dm_summary, False),
        ("6_Resource_Usage", table_6_constraints, False),
        ("7_Reallocation_Matrix", table_7_reallocation, True),
        ("8_Patient_Decisions", table_8_patients, False),
    ]
    for sheet_name, df_out, keep_idx in sheets:
        df_out.to_excel(writer, sheet_name=sheet_name, index=keep_idx)
        ws = writer.sheets[sheet_name]
        for col_cells in ws.columns:
            max_len = max(len(str(cell.value or "")) for cell in col_cells)
            ws.column_dimensions[col_cells[0].column_letter].width = min(max_len + 4, 42)

table_8_patients.to_csv(CSV_OUT, index=False)
print(f"\nExported all results tables to '{EXCEL_OUT}' and '{CSV_OUT}'.")